In [1]:
import pandas as pd
import glob
import os

In [2]:
data = pd.read_csv("summary_data/ks_test_hits.csv", index_col=0)
positive_controls = data.uid

In [3]:
protein_data = pd.read_csv("summary_data/01.structure_hits.csv")
protein_data = protein_data.query("uid in @positive_controls")
protein_data

,Unnamed: 0,uid,Entry,rv,set,pdb_id,pdb_chain,filtered_length
8,9,Y3766_MYCTU,O69731,Rv3766,setA,NaN,NaN,261.0
23,24,Y3193_MYCTU,P9WFL3,Rv3193c,setA,NaN,NaN,918.0
24,26,Y738_MYCTU,P9WKS3,Rv0738,setA,NaN,NaN,181.0
27,29,Y3818_MYCTU,P9WH21,Rv3818,setA,NaN,NaN,508.0
28,30,Y1735_MYCTU,P9WLS5,Rv1735c,setA,NaN,NaN,60.0
...,...,...,...,...,...,...,...,...
3707,3934,O53896_MYCTU,O53896,Rv0983,setA,NaN,NaN,292.0
3711,3938,I6YGX9_MYCTU,I6YGX9,Rv3692,setA,NaN,NaN,315.0
3719,3946,Q1LVD4_MYCTU,Q1LVD4,Rv3918c,setA,NaN,NaN,276.0
3751,3981,I6YCQ4_MYCTU,I6YCQ4,Rv3670,setA,NaN,NaN,307.0


## We neeed the list of mutations analyzed, the GEO scoring data, and the protein 3D structure to download

In [4]:
!mkdir summary_data/hit_download_top50

In [5]:
## Table of structure search hits

d1 = pd.read_csv("summary_data/01A.structure_search_hits_no_hits.csv")
d2 = pd.read_csv("summary_data/01B.structure_search_hits_0p90_coverage.csv")
d3 = pd.read_csv("summary_data/01C.structure_search_hits_less_than_90_percent.csv")

d = pd.concat([d1, d2, d3])


In [6]:
#! rm -r summary_data/hit_download_top50/*


In [9]:
! mkdir summary_data/hit_download_ks_test

In [10]:
for _,row in protein_data.iterrows():
    
    
    protein=row.uid
    uid = row.uid
    
    
    if not os.path.isfile(f"output/{protein}/all_mutations_analyzed.csv"):
        print("no mutation analyzed file")
        
    if not os.path.isfile(f"output/{protein}/G_scores.csv"):
        print("no score file")
    os.system(f"mkdir summary_data/hit_download_ks_test/{protein}")
    os.system(f"cp output/{protein}/all_mutations_analyzed.csv summary_data/hit_download_ks_test/{protein}/all_mutations_analyzed.csv")
    os.system(f"cp output/{protein}/G_scores.csv summary_data/hit_download_ks_test/{protein}/G_scores.csv")
    
    os.system(f"cp filtered_distmaps/{row.Entry}* summary_data/hit_download_ks_test/{protein}/")
    
    protein_set = row.set
    
    if protein_set=='setB':
        structure_hit = d.query("uid==@protein").iloc[0]
        bit=structure_hit.bitscore
        
        try:
            protein_structure = glob.glob(
                    f"../structure_search/output/{uid}_b{bit}/compare/aux/{uid}_b{bit}_{structure_hit.pdb_id.lower()}_{structure_hit.pdb_chain}_{int(structure_hit.mapping_index)}.pdb"
                )[0]
        except:
            continue

        os.system(f"cp {protein_structure} summary_data/hit_download_ks_test/{protein}")
        
    else: # get the alphafold hit
        entry = row.Entry
        os.system(f"cp ../alphafold/UP000001584_83332_MYCTU_v4_b70_filtered/AF-{entry}-F1-model_v4_beta70filtered.pdb summary_data/hit_download_ks_test/{protein}")

In [11]:
!pwd

/n/data1/hms/dbmi/farhat/anna/structure_annotation/2024_11_13_GeO


In [9]:
!../structure_search/output/RSMG_MYCTU_b0.2/compare/aux/

/bin/bash: ../structure_search/output/RSMG_MYCTU_b0.2/compare/aux/: Is a directory
